In [1]:
# ============================================================
# CELL 1 — COLAB ENVIRONMENT CHECK
# ============================================================

import sys
import torch
import importlib.metadata

print("=" * 70)
print("COLAB TTS ENVIRONMENT")
print("=" * 70)

print("Python    :", sys.version.split()[0])
print("PyTorch   :", torch.__version__)
print("CUDA      :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU       :", torch.cuda.get_device_name(0))

print("Protobuf  :", importlib.metadata.version("protobuf"))
print("Click     :", importlib.metadata.version("click"))

print("=" * 70)

COLAB TTS ENVIRONMENT
Python    : 3.13.15
PyTorch   : 2.11.0+cu128
CUDA      : True
GPU       : Tesla T4
Protobuf  : 5.29.6
Click     : 8.5.0


In [2]:
# ============================================================
# CELL 2 — INSTALL PIPER TTS
# ============================================================
#
# Piper runs through ONNX Runtime and does NOT need us to
# replace Colab's PyTorch installation.
# ============================================================

!pip install -q piper-tts

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.1/34.1 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 89.8 MB/s eta 0:00:00


In [3]:
# ============================================================
# CELL 3 — VERIFY PIPER INSTALLATION
# ============================================================

import torch
import piper

print("=" * 70)
print("PIPER TTS CHECK")
print("=" * 70)

print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())
print("Piper   :", getattr(piper, "__version__", "installed"))

if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))

print("=" * 70)
print("✅ Piper imported successfully")

PIPER TTS CHECK
PyTorch : 2.11.0+cu128
CUDA    : True
Piper   : installed
GPU     : Tesla T4
✅ Piper imported successfully


In [4]:
# ============================================================
# CELL 4 — TTS WORKSPACE + TEST SCRIPTS
# ============================================================

from pathlib import Path
from IPython.display import Audio, display
import os
import subprocess
import glob
import wave
import shutil

BASE_DIR = Path("/content/tts_samples")

TELUGU_DIR = BASE_DIR / "telugu"
HINDI_DIR = BASE_DIR / "hindi"
ENGLISH_DIR = BASE_DIR / "english"

for directory in [
    TELUGU_DIR,
    HINDI_DIR,
    ENGLISH_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

TELUGU_TEXT = """
మన జీవితంలో మనం తీసుకునే చిన్న చిన్న నిర్ణయాలే
మన భవిష్యత్తును మార్చగలవు.

ప్రతిరోజూ ఒక కొత్త విషయం నేర్చుకోవడానికి
కొంత సమయం కేటాయిస్తే,
కొన్ని నెలల్లోనే మనలో ఎంతో మార్పు కనిపిస్తుంది.

విజయం ఒక్కరోజులో రాదు.
ప్రతిరోజూ చేసే చిన్న ప్రయత్నాలే
ఒకరోజు పెద్ద విజయంగా మారతాయి.
"""

HINDI_TEXT = """
हमारी जिंदगी में लिए गए छोटे-छोटे फैसले ही
हमारा भविष्य बदल सकते हैं।

अगर हम हर दिन कुछ नया सीखने के लिए
थोड़ा सा समय निकालें,
तो कुछ ही महीनों में हमारे अंदर
बहुत बड़ा बदलाव दिखाई दे सकता है।

सफलता एक दिन में नहीं मिलती।
हर दिन की छोटी कोशिशें ही
एक दिन बड़ी कामयाबी बन जाती हैं।
"""

ENGLISH_TEXT = """
The small decisions we make every day
can completely change our future.

If we spend even a little time
learning something new each day,
we can see a huge difference
in ourselves within just a few months.

Success does not happen overnight.
Small efforts repeated every single day
eventually create big results.
"""

print("Workspace:", BASE_DIR)
print("✅ Scripts loaded")

Workspace: /content/tts_samples
✅ Scripts loaded


In [5]:
# ============================================================
# CELL 5 — VOICE CONFIGURATION
# ============================================================

VOICES = {

    "telugu": [
        {
            "name": "maya",
            "model": "te_IN-maya-medium",
            "gender": "Female"
        },
        {
            "name": "padmavathi",
            "model": "te_IN-padmavathi-medium",
            "gender": "Female"
        },
        {
            "name": "venkatesh",
            "model": "te_IN-venkatesh-medium",
            "gender": "Male"
        }
    ],

    "hindi": [
        {
            "name": "pratham",
            "model": "hi_IN-pratham-medium",
            "gender": "Male"
        },
        {
            "name": "priyamvada",
            "model": "hi_IN-priyamvada-medium",
            "gender": "Female"
        },
        {
            "name": "rohan",
            "model": "hi_IN-rohan-medium",
            "gender": "Male"
        }
    ],

    "english": [
        {
            "name": "lessac",
            "model": "en_US-lessac-high",
            "gender": "Female"
        },
        {
            "name": "ryan",
            "model": "en_US-ryan-high",
            "gender": "Male"
        },
        {
            "name": "libritts",
            "model": "en_US-libritts-high",
            "gender": "Mixed"
        }
    ]
}

for language, voice_list in VOICES.items():

    print("\n" + "=" * 60)
    print(language.upper())

    for voice in voice_list:
        print(
            f'{voice["gender"]:8s} | '
            f'{voice["model"]}'
        )


TELUGU
Female   | te_IN-maya-medium
Female   | te_IN-padmavathi-medium
Male     | te_IN-venkatesh-medium

HINDI
Male     | hi_IN-pratham-medium
Female   | hi_IN-priyamvada-medium
Male     | hi_IN-rohan-medium

ENGLISH
Female   | en_US-lessac-high
Male     | en_US-ryan-high
Mixed    | en_US-libritts-high


In [6]:
# ============================================================
# CELL 6 — DOWNLOAD PIPER VOICE MODELS
# ============================================================

import urllib.request

PIPER_BASE = (
    "https://huggingface.co/rhasspy/piper-voices/"
    "resolve/main"
)

def download_voice_model(model_name):

    # Parse:
    # te_IN-maya-medium
    # hi_IN-pratham-medium
    # en_US-lessac-high

    parts = model_name.split("-")

    language = parts[0]
    speaker = parts[1]
    quality = "-".join(parts[2:])

    model_url = (
        f"{PIPER_BASE}/"
        f"{language[:2]}/"
        f"{language}/"
        f"{speaker}/"
        f"{quality}/"
        f"{model_name}.onnx"
    )

    config_url = (
        f"{PIPER_BASE}/"
        f"{language[:2]}/"
        f"{language}/"
        f"{speaker}/"
        f"{quality}/"
        f"{model_name}.onnx.json"
    )

    model_dir = BASE_DIR / "models" / model_name

    model_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    model_file = model_dir / f"{model_name}.onnx"
    config_file = model_dir / f"{model_name}.onnx.json"

    if not model_file.exists():

        print(f"Downloading {model_name}.onnx ...")

        urllib.request.urlretrieve(
            model_url,
            model_file
        )

    else:

        print(f"Already exists: {model_file.name}")

    if not config_file.exists():

        print(f"Downloading {model_name}.onnx.json ...")

        urllib.request.urlretrieve(
            config_url,
            config_file
        )

    return model_file, config_file

In [7]:
# ============================================================
# CELL 7 — DOWNLOAD TELUGU + HINDI + ENGLISH VOICES
# ============================================================

ALL_VOICES = []

for language, voice_list in VOICES.items():

    for voice in voice_list:

        voice_copy = voice.copy()
        voice_copy["language"] = language

        ALL_VOICES.append(
            voice_copy
        )

for voice in ALL_VOICES:

    print(
        "\n" +
        "=" * 70
    )

    print(
        voice["language"].upper(),
        "|",
        voice["model"]
    )

    try:

        model_file, config_file = (
            download_voice_model(
                voice["model"]
            )
        )

        voice["model_file"] = str(model_file)
        voice["config_file"] = str(config_file)

        print("✅ Ready")

    except Exception as e:

        print("❌ FAILED")
        print(e)


TELUGU | te_IN-maya-medium
✅ Ready

TELUGU | te_IN-padmavathi-medium
✅ Ready

TELUGU | te_IN-venkatesh-medium
✅ Ready

HINDI | hi_IN-pratham-medium
✅ Ready

HINDI | hi_IN-priyamvada-medium
✅ Ready

HINDI | hi_IN-rohan-medium
✅ Ready

ENGLISH | en_US-lessac-high
✅ Ready

ENGLISH | en_US-ryan-high
✅ Ready

ENGLISH | en_US-libritts-high
✅ Ready


In [8]:
# ============================================================
# CELL 8 — VERIFY MODELS
# ============================================================

for voice in ALL_VOICES:

    model_file = Path(
        voice.get("model_file", "")
    )

    config_file = Path(
        voice.get("config_file", "")
    )

    print(
        f'{voice["language"]:8s} | '
        f'{voice["model"]:28s} | '
        f'Model: {model_file.exists()} | '
        f'Config: {config_file.exists()}'
    )

telugu   | te_IN-maya-medium            | Model: True | Config: True
telugu   | te_IN-padmavathi-medium      | Model: True | Config: True
telugu   | te_IN-venkatesh-medium       | Model: True | Config: True
hindi    | hi_IN-pratham-medium         | Model: True | Config: True
hindi    | hi_IN-priyamvada-medium      | Model: True | Config: True
hindi    | hi_IN-rohan-medium           | Model: True | Config: True
english  | en_US-lessac-high            | Model: True | Config: True
english  | en_US-ryan-high              | Model: True | Config: True
english  | en_US-libritts-high          | Model: True | Config: True


In [9]:
# ============================================================
# CELL 9 — PIPER GENERATION FUNCTION
# ============================================================

def synthesize_piper(
    text,
    model_file,
    output_file,
    length_scale=1.0
):

    command = [
        "python",
        "-m",
        "piper",
        "--model",
        str(model_file),
        "--output_file",
        str(output_file),
        "--length_scale",
        str(length_scale)
    ]

    process = subprocess.run(
        command,
        input=text,
        text=True,
        capture_output=True
    )

    if process.returncode != 0:

        print(process.stdout)
        print(process.stderr)

        raise RuntimeError(
            "Piper synthesis failed."
        )

    return output_file

In [10]:
# ============================================================
# CELL 10 — GENERATE TELUGU SAMPLES
# ============================================================

for voice in ALL_VOICES:

    if voice["language"] != "telugu":
        continue

    output_file = (
        TELUGU_DIR /
        f'{voice["model"]}.wav'
    )

    print(
        f"\nGenerating: {voice['model']}"
    )

    synthesize_piper(
        text=TELUGU_TEXT,
        model_file=voice["model_file"],
        output_file=output_file,
        length_scale=1.0
    )

    print(
        "✅",
        output_file
    )



Generating: te_IN-maya-medium
✅ /content/tts_samples/telugu/te_IN-maya-medium.wav

Generating: te_IN-padmavathi-medium
✅ /content/tts_samples/telugu/te_IN-padmavathi-medium.wav

Generating: te_IN-venkatesh-medium
✅ /content/tts_samples/telugu/te_IN-venkatesh-medium.wav


In [11]:
# ============================================================
# CELL 11 — TELUGU VOICE AUDITION
# ============================================================

for file in sorted(
    TELUGU_DIR.glob("*.wav")
):

    print("\n" + "=" * 70)
    print(file.stem)
    print("=" * 70)

    display(
        Audio(
            filename=str(file),
            autoplay=False
        )
    )

Output hidden; open in https://colab.research.google.com to view.

In [12]:
# ============================================================
# CELL 12 — GENERATE HINDI SAMPLES
# ============================================================

for voice in ALL_VOICES:

    if voice["language"] != "hindi":
        continue

    output_file = (
        HINDI_DIR /
        f'{voice["model"]}.wav'
    )

    print(
        f"\nGenerating: {voice['model']}"
    )

    synthesize_piper(
        text=HINDI_TEXT,
        model_file=voice["model_file"],
        output_file=output_file,
        length_scale=1.0
    )

    print(
        "✅",
        output_file
    )


Generating: hi_IN-pratham-medium
✅ /content/tts_samples/hindi/hi_IN-pratham-medium.wav

Generating: hi_IN-priyamvada-medium
✅ /content/tts_samples/hindi/hi_IN-priyamvada-medium.wav

Generating: hi_IN-rohan-medium
✅ /content/tts_samples/hindi/hi_IN-rohan-medium.wav


In [13]:
# ============================================================
# CELL 13 — HINDI VOICE AUDITION
# ============================================================

for file in sorted(
    HINDI_DIR.glob("*.wav")
):

    print("\n" + "=" * 70)
    print(file.stem)
    print("=" * 70)

    display(
        Audio(
            filename=str(file),
            autoplay=False
        )
    )

Output hidden; open in https://colab.research.google.com to view.

In [14]:
# ============================================================
# CELL 14 — GENERATE ENGLISH SAMPLES
# ============================================================

for voice in ALL_VOICES:

    if voice["language"] != "english":
        continue

    output_file = (
        ENGLISH_DIR /
        f'{voice["model"]}.wav'
    )

    print(
        f"\nGenerating: {voice['model']}"
    )

    synthesize_piper(
        text=ENGLISH_TEXT,
        model_file=voice["model_file"],
        output_file=output_file,
        length_scale=1.0
    )

    print(
        "✅",
        output_file
    )


Generating: en_US-lessac-high
✅ /content/tts_samples/english/en_US-lessac-high.wav

Generating: en_US-ryan-high
✅ /content/tts_samples/english/en_US-ryan-high.wav

Generating: en_US-libritts-high
✅ /content/tts_samples/english/en_US-libritts-high.wav


In [15]:
# ============================================================
# CELL 15 — ENGLISH VOICE AUDITION
# ============================================================

for file in sorted(
    ENGLISH_DIR.glob("*.wav")
):

    print("\n" + "=" * 70)
    print(file.stem)
    print("=" * 70)

    display(
        Audio(
            filename=str(file),
            autoplay=False
        )
    )

Output hidden; open in https://colab.research.google.com to view.

In [16]:
# ============================================================
# CELL 16 — MEASURE AUDIO DURATIONS
# ============================================================

def get_wav_duration(file):

    with wave.open(str(file), "rb") as wav:

        frames = wav.getnframes()
        rate = wav.getframerate()

        return frames / float(rate)


for directory in [
    TELUGU_DIR,
    HINDI_DIR,
    ENGLISH_DIR
]:

    print("\n" + "=" * 70)
    print(directory.name.upper())
    print("=" * 70)

    for file in sorted(
        directory.glob("*.wav")
    ):

        duration = get_wav_duration(file)

        print(
            f"{file.stem:35s} "
            f"{duration:6.2f} sec"
        )


TELUGU
te_IN-maya-medium                    23.60 sec
te_IN-padmavathi-medium              23.20 sec
te_IN-venkatesh-medium               17.74 sec

HINDI
hi_IN-pratham-medium                 18.91 sec
hi_IN-priyamvada-medium              23.14 sec
hi_IN-rohan-medium                   24.16 sec

ENGLISH
en_US-lessac-high                    18.22 sec
en_US-libritts-high                  18.63 sec
en_US-ryan-high                      16.04 sec
